# Difusión del calor 2D en un rectángulo

Notebook de apoyo para visualizar la solución analítica truncada de la ecuación de difusión del calor en el rectángulo

$$0<x<a,\qquad 0<y<b,$$

con condiciones homogéneas de Dirichlet en todo el borde:

$$\psi(0,y,t)=\psi(a,y,t)=\psi(x,0,t)=\psi(x,b,t)=0.$$

La solución evaluada es

$$
\psi(x,y,t)=\sum_{n=1}^{N}\sum_{m=1}^{M}
A_{nm}\sin\left(\frac{n\pi x}{a}\right)
\sin\left(\frac{m\pi y}{b}\right)
\exp\left[-\alpha\pi^2\left(\frac{n^2}{a^2}+\frac{m^2}{b^2}\right)t\right].
$$

Este notebook corresponde al material computacional del profesor **Guillermo Rubilar**, adaptado aquí para quedar integrado al repositorio de ayudantías.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib import animation
from IPython.display import HTML

## Parámetros del problema

Se toma un rectángulo de lados $a$ y $b$, difusividad térmica $\alpha$ y una fuente inicial localizada cerca de $(x_0,y_0)$. La condición inicial puntual se representa mediante los coeficientes modales de una delta de Dirac bidimensional truncada.

In [ ]:
# Geometría y parámetros físicos
a = 1.0
b = 2.0
alpha = 1.0
T0 = 1.0

# Posición de la fuente inicial localizada
x0 = 0.6 * a
y0 = 0.7 * b

# Número de modos retenidos en la suma doble
nmax = 100
mmax = 100

# Malla espacial
Nx = 140
Ny = 220
x = np.linspace(0, a, Nx)
y = np.linspace(0, b, Ny)
X, Y = np.meshgrid(x, y, indexing='xy')

## Coeficientes modales

Para una condición inicial idealizada como

$$
\varphi(x,y)=T_0\delta(x-x_0)\delta(y-y_0),
$$

los coeficientes de la serie doble de senos quedan dados por

$$
A_{nm}=4T_0\sin\left(\frac{n\pi x_0}{a}\right)\sin\left(\frac{m\pi y_0}{b}\right).
$$

In [ ]:
def A_nm(n, m):
    """Coeficiente modal para una fuente puntual idealizada."""
    return 4.0 * T0 * np.sin(n * np.pi * x0 / a) * np.sin(m * np.pi * y0 / b)


def psi(x_grid, y_grid, t, nmax=nmax, mmax=mmax):
    """Evalúa la solución analítica truncada en la malla (x_grid,y_grid)."""
    total = np.zeros_like(x_grid, dtype=float)
    for n in range(1, nmax + 1):
        sx = np.sin(n * np.pi * x_grid / a)
        for m in range(1, mmax + 1):
            sy = np.sin(m * np.pi * y_grid / b)
            decay = np.exp(-alpha * np.pi**2 * (n**2 / a**2 + m**2 / b**2) * t)
            total += A_nm(n, m) * sx * sy * decay
    return total

## Visualización 2D

La vista 2D representa $\psi(x,y,t)$ como mapa de calor sobre el rectángulo.

In [ ]:
def grafico_2d(t=0.0, nmodos=100, mmodos=100):
    Z = psi(X, Y, t, nmax=nmodos, mmax=mmodos)
    fig, ax = plt.subplots(figsize=(5, 7))
    im = ax.imshow(
        Z,
        extent=[0, a, 0, b],
        origin='lower',
        aspect='equal'
    )
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_title(rf'$\psi(x,y,t)$, $t={t:.3f}$')
    fig.colorbar(im, ax=ax, label=r'$\psi$')
    plt.show()

grafico_2d(t=0.0, nmodos=100, mmodos=100)

## Visualización 3D

La vista 3D representa la temperatura como altura sobre el plano $xy$.

In [ ]:
def grafico_3d(t=0.0, nmodos=100, mmodos=100):
    Z = psi(X, Y, t, nmax=nmodos, mmax=mmodos)
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X, Y, Z, cmap='turbo', linewidth=0, antialiased=True)
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_zlabel(r'$\psi$')
    ax.set_title(rf'$\psi(x,y,t)$, $t={t:.3f}$')
    fig.colorbar(surf, ax=ax, shrink=0.65, label=r'$\psi$')
    plt.show()

grafico_3d(t=0.0, nmodos=100, mmodos=100)

## Animación 2D

Para una demostración en vivo puede convenir bajar `nmodos` y `mmodos`, porque el costo crece como $N M$ por cada instante de tiempo.

In [ ]:
def animacion_2d(tmax=0.08, frames=50, nmodos=50, mmodos=50):
    tiempos = np.linspace(0, tmax, frames)
    Z0 = psi(X, Y, tiempos[0], nmax=nmodos, mmax=mmodos)

    fig, ax = plt.subplots(figsize=(5, 7))
    im = ax.imshow(Z0, extent=[0, a, 0, b], origin='lower', aspect='equal')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    title = ax.set_title(rf'$\psi(x,y,t)$, $t={tiempos[0]:.3f}$')
    fig.colorbar(im, ax=ax, label=r'$\psi$')

    def update(k):
        Z = psi(X, Y, tiempos[k], nmax=nmodos, mmax=mmodos)
        im.set_data(Z)
        im.set_clim(vmin=np.min(Z), vmax=np.max(Z))
        title.set_text(rf'$\psi(x,y,t)$, $t={tiempos[k]:.3f}$')
        return im, title

    ani = animation.FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())

# Ejecutar esta línea para ver la animación en Jupyter:
# animacion_2d(tmax=0.08, frames=50, nmodos=50, mmodos=50)

## Exportación opcional

Para guardar un GIF o MP4, conviene construir explícitamente un `FuncAnimation` y usar `ani.save(...)` con el escritor disponible en el entorno local.